# VLM на открытых данных VK — Ноутбук 01: окружение и разведка данных

Учебный проект: дообучение Vision-Language модели на открытых данных коллекции
`deepvk` и оценка на бенчмарках **GQA-ru** и **MMBench-ru**.

Всё обучение и инференс выполняются в **Google Colab** (бесплатный тариф, GPU T4, 16 ГБ VRAM).

**Что делает этот ноутбук (шаги 1–2 плана):**
1. Устанавливает все зависимости.
2. Логинится в HuggingFace через `notebook_login()`.
3. Скачивает и изучает три датасета — выводит структуру (schema), примеры и размеры:
   - `deepvk/LLaVA-Instruct-ru` — 144k инструктивных примеров для дообучения;
   - `deepvk/GQA-ru` — бенчмарк GQA (~80k);
   - `deepvk/MMBench-ru` — бенчмарк MMBench (~3.9k).

> Модель **не** загружается и обучение **не** запускается — это отдельные ноутбуки.

---

> **Совет по GPU в Colab:** для этого ноутбука GPU не обязателен (только скачивание/просмотр данных),
> но включить его не помешает: `Runtime → Change runtime type → T4 GPU`.

## 1. Установка зависимостей

In [ ]:
# Полный набор зависимостей проекта (нужен для последующих ноутбуков обучения/оценки).
# Для самой разведки данных достаточно datasets/huggingface_hub/pillow, остальное ставим сразу,
# чтобы окружение было готово целиком.
!pip install -q -U datasets huggingface_hub
!pip install -q -U transformers peft accelerate bitsandbytes evaluate
# ВАЖНО: Pillow пинуем на <12. Свежий Pillow 12 в Colab ломает импорт transformers
# (ImportError: cannot import name '_Ink' from 'PIL._typing'). Ставим последним, чтобы победил.
!pip install -q -U "pillow<12"
print('\n[OK] Зависимости установлены. Если Colab предложит перезапуск сессии — согласитесь.')

In [ ]:
# Проверяем версии ключевых библиотек
import transformers, datasets, peft, accelerate, huggingface_hub
import PIL, sys
print('python           :', sys.version.split()[0])
print('transformers     :', transformers.__version__)
print('datasets         :', datasets.__version__)
print('peft             :', peft.__version__)
print('accelerate       :', accelerate.__version__)
print('huggingface_hub  :', huggingface_hub.__version__)
print('pillow           :', PIL.__version__)
try:
    import bitsandbytes as bnb
    print('bitsandbytes     :', bnb.__version__)
except Exception as e:
    print('bitsandbytes     : не импортировался (ок для разведки данных):', e)

import torch
print('torch            :', torch.__version__)
print('CUDA доступна     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU              :', torch.cuda.get_device_name(0))

## 2. Логин в HuggingFace

Нужен токен с **read**-доступом: HuggingFace → Settings → Access Tokens.
После запуска ячейки вставьте токен в появившееся поле.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Проверяем, что логин прошёл
from huggingface_hub import whoami
try:
    user = whoami()
    print('[OK] Залогинены как:', user['name'])
except Exception as e:
    print('[!] Логин не выполнен, повторите ячейку с notebook_login():', e)

## 3. Разведка датасетов

**Подход.** Датасеты содержат изображения и довольно объёмны
(`LLaVA-Instruct-ru` ~144k, `GQA-ru` ~80k). Чтобы не тянуть десятки ГБ на диск Colab
ради простого осмотра, мы:

- берём **метаданные** (schema, число примеров, размеры) через `load_dataset_builder` —
  это не качает сами данные;
- **потоково** (`streaming=True`) подгружаем по несколько первых примеров каждого сплита,
  чтобы посмотреть реальные поля и картинки.

Так мы подтверждаем, что доступ к данным работает и структура понятна, не забивая диск.
Полное скачивание нужной подвыборки будет в ноутбуке обучения.

In [ ]:
import itertools
from datasets import (load_dataset, load_dataset_builder,
                      get_dataset_config_names, get_dataset_split_names)
from PIL import Image
import matplotlib.pyplot as plt


def show_schema_and_sizes(dataset_id):
    """Метаданные без скачивания: конфиги, schema, число примеров и размеры."""
    print('#' * 78)
    print('#', dataset_id)
    print('#' * 78)
    try:
        configs = get_dataset_config_names(dataset_id)
    except Exception as e:
        print('  (не удалось получить список конфигов:', e, ')')
        configs = [None]
    print('Конфиги:', configs)
    cfg = configs[0] if configs else None

    b = load_dataset_builder(dataset_id, cfg)
    info = b.info
    if info.description:
        print('\nОписание:', info.description.strip()[:400])
    print('\nSchema (features):')
    for name, feat in (info.features or {}).items():
        print(f'    {name:20s} -> {feat}')
    print('\nСплиты и размеры:')
    total = 0
    if info.splits:
        for name, s in info.splits.items():
            total += s.num_examples
            mb = (s.num_bytes or 0) / 1e6
            print(f'    {name:12s} : {s.num_examples:>8,} примеров   ({mb:8.1f} MB)')
    print(f'    {"ИТОГО":12s} : {total:>8,} примеров')
    if info.download_size:
        print(f'\nРазмер загрузки (download_size): {info.download_size/1e6:,.1f} MB')
    if info.dataset_size:
        print(f'Размер в распакованном виде     : {info.dataset_size/1e6:,.1f} MB')
    print()
    return cfg


def peek_examples(dataset_id, cfg=None, n=3, show_image=True):
    """Потоково берём первые n примеров первого сплита и печатаем поля."""
    ds = load_dataset(dataset_id, cfg, streaming=True)
    split_name = list(ds.keys())[0]
    print(f'--- {dataset_id}  (сплит: {split_name}) — первые {n} примеров ---')
    first_image = None
    for i, ex in enumerate(itertools.islice(ds[split_name], n)):
        print(f'\n[пример {i}]')
        for k, v in ex.items():
            if isinstance(v, Image.Image):
                print(f'  {k:15s}: <PIL.Image {v.mode} {v.size}>')
                if first_image is None:
                    first_image = v
            else:
                s = repr(v)
                print(f'  {k:15s}: {s[:300]}{"..." if len(s) > 300 else ""}')
    if show_image and first_image is not None:
        plt.figure(figsize=(4, 4))
        plt.imshow(first_image)
        plt.axis('off')
        plt.title(f'{dataset_id}: пример изображения')
        plt.show()
    print()

### 3.1 `deepvk/LLaVA-Instruct-ru` — данные для дообучения

In [ ]:
cfg_llava = show_schema_and_sizes('deepvk/LLaVA-Instruct-ru')

In [ ]:
peek_examples('deepvk/LLaVA-Instruct-ru', cfg_llava, n=2)

### 3.2 `deepvk/GQA-ru` — бенчмарк 1

In [ ]:
cfg_gqa = show_schema_and_sizes('deepvk/GQA-ru')

In [ ]:
peek_examples('deepvk/GQA-ru', cfg_gqa, n=3)

### 3.3 `deepvk/MMBench-ru` — бенчмарк 2

In [ ]:
cfg_mmb = show_schema_and_sizes('deepvk/MMBench-ru')

In [ ]:
peek_examples('deepvk/MMBench-ru', cfg_mmb, n=3)

## 4. Итог и следующие шаги

Если все три ячейки разведки отработали без ошибок — доступ к данным есть,
структура и размеры понятны. Зафиксируйте в `results/` (например, скриншот вывода
или скопированный лог) для отчёта.

**Дальше (отдельные ноутбуки, только после подтверждения):**
- `02` — загрузка базовой модели `deepvk/llava-gemma-2b-lora`, проверка инференса и влезаемости QLoRA в 16 ГБ;
- zero-shot оценка на GQA-ru / MMBench-ru;
- LoRA-дообучение на подвыборке `LLaVA-Instruct-ru` с чекпоинтами в Google Drive;
- повторная оценка и сравнение метрик до/после.

> Замечание: если какой-то ID не грузится — не менять план молча, а зафиксировать ошибку
> и обсудить альтернативу (`deepvk/llava-saiga-8b` как запасная модель).